# CPU collective 与控制依赖

R08 的可执行参考。2 个逻辑 CPU 设备，比较同步 psum、显式 async pair 和同步控制依赖。这里复现数值与 IR，不测量 TPU overlap。详细失败边界见 [说明](overlap-and-scheduling.md)。

In [1]:
from pathlib import Path
import json, sys, subprocess
from datetime import datetime, timezone
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "upstream-sources.lock").is_file())
HERE = ROOT / "research/software-stack"
sys.path.insert(0, str(HERE))
report = json.loads((HERE / "overlap-results.json").read_text())
print(report["evidence_level"], report["qualifiers"])
print(json.dumps(report["topology"], ensure_ascii=False, indent=2))

RUN-CPU ['VERSION-SKEW']
{
  "backend": "cpu",
  "process_count": 1,
  "device_count": 2,
  "devices": [
    "cpu:0",
    "cpu:1"
  ],
  "mesh_axes": {
    "d": 2
  },
  "x_global_shape": [
    128,
    64
  ],
  "x_local_shape": [
    64,
    64
  ],
  "w_replicated_shape": [
    64,
    64
  ],
  "local_expression": "psum(x, 'd') + w @ w",
  "output_global_shape": [
    128,
    64
  ],
  "scope": "Two logical CPU devices on one host; no TPU or inter-host network."
}


输入 X[128,64] 分片为两份 x[64,64]，W[64,64] 复制。局部输出为 psum(x)+W@W，按第 0 维拼接回 [128,64]。独立 W@W 的 VMA 与 Future 一致；x@W 组合的失败单独保存。

In [2]:
for case in report["cases"]:
    print(case["case"], "max_error=", case["max_absolute_error"])
    print("  initial targets:", case["stablehlo_targets"])
    print("  control edge:", case["control_edge_after_rewriter"])
for failure in report["expected_failures"]:
    print(failure["case"], failure["phase"], failure["error_fragment"])

sync max_error= 1.460352109239338e-07
  initial targets: {}
  control edge: None
async max_error= 1.460352109239338e-07
  initial targets: {'all-reduce-start': 1, 'all-reduce-done': 1}
  control edge: None
sync-control max_error= 1.460352109239338e-07
  initial targets: {'control_dep': 1}
  control edge: {'source': 'dot_general.2', 'target': 'psum_invariant.6', 'source_opcode': 'kDot', 'target_opcode': 'kAllReduce'}
scheduled-varying-checked tracing requires varying manual axes to match
scheduled-varying-unchecked tracing has no attribute done
scheduled-invariant-default-layout MLIR-verification incorrect layout dense<>
explicit-layout-backend-error CPU-native-compilation is live and cannot be removed


下面在新 Python 进程中重新执行完整实验，确保双 CPU 配置在 import JAX 前生效。每次创建新的忽略目录，保留原始产物。四个预期错误会被捕获；意外错误仍令运行失败。

In [3]:
capture = ROOT / "artifacts/jax-stack" / ("overlap-notebook-" + datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S%f"))
command = [str(ROOT / ".venv/bin/python"), "-B", str(HERE / "overlap_probe.py"), "--output", str(capture)]
run = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, timeout=180)
if run.returncode:
    raise RuntimeError(run.stdout + run.stderr)
print(run.stdout.strip())
print("capture:", capture.relative_to(ROOT))

{"capture": "overlap-notebook-20260914161636905717", "cases": 3, "qualifiers": ["VERSION-SKEW"]}
capture: artifacts/jax-stack/overlap-notebook-20260914161636905717


In [4]:
from verify_overlap import verify
result = verify(capture)
print(json.dumps({"artifacts": result["artifact_count"], "cases": len(result["cases"]), "expected_failures": len(result["expected_failures"]), "qualifiers": result["qualifiers"]}, indent=2))
assert all(c["max_absolute_error"] < 2e-5 for c in result["cases"])
print("数值、字节、有效 IR、async 改写与同步控制边复查通过")

{
  "artifacts": 593,
  "cases": 3,
  "expected_failures": 4,
  "qualifiers": [
    "VERSION-SKEW"
  ]
}
数值、字节、有效 IR、async 改写与同步控制边复查通过


Native 行为属于当前 jaxlib wheel，仍为 VERSION-SKEW。CPU 将显式 async collective 转回同步；成功的同步控制边也不能证明实际重叠。异步控制路径需要自建 wheel 复验；TPU/libtpu、设备 trace 与目标运行时间仍待验证。